# AI-Powered Voice Phishing (Vishing) Detection
### End-to-End Walkthrough: Multi-Modal Acoustic & Lexical Telephony Defense

This tutorial notebook demonstrates the complete detection pipeline:
1. **Audio Ingestion & Preprocessing** (PyDub: 16 kHz Mono, Loudness Normalization, Silence Trimming)
2. **Acoustic Feature Profiling** (Librosa: 119 dimensions including MFCCs, Chroma, Spectral features, YIN pitch, and prosody)
3. **Unsupervised Anomaly Detection** (IsolationForest trained on benign speech)
4. **Offline Speech Recognition** (Vosk / Kaldi ASR)
5. **Scam Cue NLP Analyzer** (YAML lexicon regex matching & category severity weights)
6. **Multi-Modal Risk Scorer** (Fused 0-100 Threat Index with EMA smoothing and forensic explanations)

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Ensure project src is in path
sys.path.append(str(Path("..").resolve() / "src"))

from vishing_detector.audio.loader import AudioLoader
from vishing_detector.features.acoustic import AcousticFeatureExtractor
from vishing_detector.anomaly.detector import AcousticAnomalyDetector
from vishing_detector.asr.transcriber import get_transcriber
from vishing_detector.nlp.scam_cues import ScamCueAnalyzer
from vishing_detector.fusion.scorer import RiskScorer
from vishing_detector.pipeline import VishingDetectionPipeline

print("All modules imported successfully!")

## 1. Audio Ingestion & Preprocessing
We ingest an audio recording, convert to 16 kHz mono WAV, normalize volume to -20 dBFS, and split into overlapping 3-second windows with 1-second hops to simulate real-time telephony streaming.

In [2]:
audio_file = Path("../data/samples/sample_scam.wav")
loader = AudioLoader(target_sample_rate=16000, window_duration_sec=3.0, hop_duration_sec=1.0)

segment, samples = loader.load(audio_file)
chunks = loader.chunk_all(segment)

print(f"Audio duration: {len(segment)/1000.0:.2f} seconds")
print(f"Sample rate: {segment.frame_rate} Hz, Channels: {segment.channels}")
print(f"Generated streaming chunks: {len(chunks)}")

## 2. Acoustic Feature Extraction
Using Librosa, we extract 119 acoustic features for each streaming chunk, capturing timbre, spectral distribution, pitch inflection, and prosodic pacing.

In [3]:
extractor = AcousticFeatureExtractor()
chunk_features = [extractor.extract_chunk(c) for c in chunks]
df_features = extractor.extract_chunks(chunks)

print(f"Extracted feature matrix shape: {df_features.shape}")
df_features[['start_time', 'end_time', 'mfcc_1_mean', 'spectral_centroid_mean', 'pitch_f0_mean', 'pause_ratio']].head()

## 3. Unsupervised Acoustic Anomaly Detection
We score each audio chunk using an Isolation Forest trained on benign conversations to detect unnatural acoustic stress or synthesized speech patterns.

In [4]:
model_path = Path("../models/acoustic_anomaly_model.joblib")
if model_path.exists():
    detector = AcousticAnomalyDetector.load(model_path)
else:
    # Fallback fit if model was not pre-saved
    detector = AcousticAnomalyDetector(feature_names=extractor.feature_names)
    detector.fit(df_features[extractor.feature_names].values)

anomaly_results = [detector.predict_chunk(f.vector) for f in chunk_features]
print(f"Anomaly score for Chunk 0: {anomaly_results[0].anomaly_score:.3f} (Threshold: {detector.calibrated_threshold:.3f})")
print("Top deviating acoustic features:", anomaly_results[0].top_deviating_features)

## 4. Speech Recognition & Scam Cue NLP Analysis
We transcribe speech and apply the configurable regex lexicon (`config/scam_lexicon.yaml`) to identify urgency, credential demands (OTP/PIN), authority impersonation, and payment demands.

In [5]:
nlp = ScamCueAnalyzer(lexicon_path="../config/scam_lexicon.yaml")
sample_text = "This is Officer Miller from the Internal Revenue Service. A legal arrest warrant has been issued. You must immediately pay with Apple gift cards or read back your OTP code."
analysis = nlp.analyze_text(sample_text)

print(f"Text threat score: {analysis.combined_text_score:.3f}")
print("Triggered categories:", analysis.summary_tags)
for c in analysis.cue_matches:
    print(f"  - [{c.category_name}] matched: '{c.matched_text}' (weight: {c.weight})")

## 5. End-to-End Streaming Simulation & Threat Fusion
Now we run the complete pipeline simulating real-time call telemetry.

In [6]:
pipeline = VishingDetectionPipeline(config_path="../config/default.yaml", asr_backend="mock")
pipeline.transcriber.predefined_script = sample_text

result = pipeline.process_file(audio_file)
expl = result.explanation

print(f"Overall Threat Score: {expl.overall_risk_score} / 100 ({expl.risk_level})")
print(f"Time to First Alert: {expl.first_alert_time}s")
print(f"Recommended Action: {expl.recommended_action}")

print("\nPrimary Forensic Indicators:")
for r in expl.primary_reasons:
    print(f"  • {r}")

## 6. Threat Progression Visualization
We plot the streaming Threat Index trajectory across time with decision threshold bands.

In [7]:
times = [a.start_time for a in result.chunk_assessments]
scores = [a.smoothed_risk for a in result.chunk_assessments]

plt.figure(figsize=(10, 4))
plt.axhspan(0, 25, color='green', alpha=0.1, label='Safe')
plt.axhspan(25, 50, color='yellow', alpha=0.15, label='Elevated')
plt.axhspan(50, 75, color='orange', alpha=0.2, label='High Threat')
plt.axhspan(75, 100, color='red', alpha=0.25, label='Critical Attack')

plt.plot(times, scores, color='crimson', lw=2.5, marker='o', label='Fused Threat Index')
plt.xlabel('Call Duration (seconds)')
plt.ylabel('Risk Score (0 - 100)')
plt.title('Real-Time Vishing Threat Progression')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()